# Week 8 Capstone — Meeting-to-Action Assistant

This notebook is the submission entry point for the COSC 650 capstone. It takes an existing meeting transcript and produces a structured operational brief with:

- executive summary
- decisions
- action items and owners
- blockers
- follow-up questions

## Rubric map

| Criterion | Evidence |
| --- | --- |
| Integration by Reuse | Week 3 structured prompting, Week 4 validation, Week 5 MiniLM/FAISS RAG, Week 6 cross-encoder reranking, Week 7 dynamic few-shot retrieval |
| Evaluation | 12 held-out cases, deterministic metrics, structured-output validity, latency, and Gemini judge on correctness, grounding, and operational usefulness |
| Required Failure Mode | A/B/C ablation, retrieval logs, component trace, and measured mitigation |
| README and Architecture | `project/README.md` |
| Portfolio Curation | polished project README, research note, and coherent Week 1–8 repository history |

The project intentionally excludes transcription, Teams/Zoom integration, automatic email/task execution, fine-tuning, deployment, and a web UI.

## Architecture

```text
existing transcript
      |
      v
MiniLM + FAISS retrieval
      |
      v
cross-encoder reranking
      |
      +---- optional dynamic examples
      |
      v
Gemini Flash structured extraction
      |
      v
deterministic validation
      |
      v
operational brief + evaluation
```

### Reuse

- **Week 3:** structured prompt/output discipline
- **Week 4:** schema validation and structured failure handling
- **Week 5:** local embeddings + FAISS retrieval
- **Week 6:** cross-encoder reranking
- **Week 7:** dynamic few-shot selection and held-out evaluation discipline

### API-budget design

The assignment requires free-tier use and reproducibility. Gemini responses are therefore persisted in `project/data/response_cache.json`. A complete uncached run is capped at:

- 36 synthesis calls = 12 held-out cases × 3 ablations
- 36 judge calls = 12 held-out cases × 3 ablations

Subsequent reruns use the cache instead of repeating those calls.

In [ ]:
# If needed in Colab:
# %pip install -q sentence-transformers faiss-cpu openai pandas numpy

import os, json, pathlib
import pandas as pd
import numpy as np

assert pathlib.Path("capstone_core.py").exists(), "Run this notebook from the project/ directory."
from capstone_cached import *

print("Held-out cases:", len(EVAL_CASES))
print("Gemini synthesis model:", GEMINI_MODEL)
print("Gemini judge model:", GEMINI_JUDGE_MODEL)
print("Embedding model:", EMBED_MODEL)
print("Reranker:", RERANK_MODEL)
print("Cache:", cache_summary())
assert 10 <= len(EVAL_CASES) <= 20

## Held-out evaluation and ablation

The same 12 held-out meeting cases are evaluated under:

- **A — transcript only**
- **B — transcript + retrieved/reranked operational context**
- **C — transcript + retrieved/reranked context + dynamic few-shot examples**

Concrete metrics:

- decision precision / recall / F1
- action-item precision / recall / F1
- owner accuracy
- blocker F1
- follow-up-question F1
- structured-output validity
- evidence support
- latency

In [ ]:
if not os.getenv("GEMINI_API_KEY"):
    print("API-BLOCKED: add GEMINI_API_KEY to Colab Secrets or the environment.")
    ABLATION_RESULTS, METRICS = {}, pd.DataFrame()
else:
    ABLATION_RESULTS, METRICS = run_ablation()
    display(METRICS.round(3))
    metric_cols = [
        "decision_precision","decision_recall","decision_f1",
        "action_precision","action_recall","action_f1",
        "owner_accuracy","blocker_f1","question_f1",
        "valid","evidence_support","latency_s"
    ]
    ABLATION_SUMMARY = METRICS.groupby("config")[metric_cols].mean().round(3)
    display(ABLATION_SUMMARY)
    print("Cache after synthesis:", cache_summary())

## LLM-as-judge

All 12 held-out cases are judged on three named dimensions:

1. **correctness**
2. **grounding**
3. **operational usefulness**

Gemini Flash is the default judge to stay within the assignment's free-tier guidance. The judge outputs are also cached.

In [ ]:
if not ABLATION_RESULTS:
    print("Run the ablation first.")
else:
    JUDGE_DF = run_judge(ABLATION_RESULTS)
    display(JUDGE_DF)
    JUDGE_SUMMARY = JUDGE_DF.groupby("config")[["correctness","grounding","operational_usefulness"]].mean().round(3)
    display(JUDGE_SUMMARY)
    print("Cache after judging:", cache_summary())

## Required failure mode and component trace

The declared failure is a suggestion or discussion point being incorrectly promoted to a confirmed decision.

The trace uses the ablation to attribute the failure:

1. If C regresses relative to B, the dynamic few-shot context is implicated.
2. Else if B regresses relative to A, retrieved/reranked context is implicated.
3. Otherwise, the lowest-scoring case is attributed to the extraction stage rather than inventing a retrieval failure.

The selected case prints the transcript, gold answer, outputs, scores, retrieved rules, and retrieved examples.

In [ ]:
if not ABLATION_RESULTS:
    FAILURE = None
    print("Run the ablation first.")
else:
    FAILURE = trace_failure(ABLATION_RESULTS)
    cid = FAILURE["case_id"]
    print("Failure case:", cid)
    print("Responsible component:", FAILURE["component"])
    print("\nTranscript:\n", FAILURE["case"]["transcript"])
    print("\nGold:\n", json.dumps(FAILURE["case"]["gold"], indent=2))
    for cfg in ["A","B","C"]:
        print(f"\n--- {cfg} ---")
        print(json.dumps(ABLATION_RESULTS[cfg][cid]["brief"], indent=2))
        print("scores:", FAILURE["scores"][cfg][cid])
        if cfg in ("B","C"):
            print("retrieved rules:")
            for x in ABLATION_RESULTS[cfg][cid]["retrieval"]["rules"]:
                print(x["doc"]["id"], round(x["rerank_score"],3), x["doc"]["text"])
        if cfg == "C":
            print("retrieved examples:")
            for x in ABLATION_RESULTS[cfg][cid]["retrieval"]["examples"]:
                print(x["doc"]["label"], round(x["rerank_score"],3), x["doc"]["text"])

## Mitigation

A narrow deterministic guardrail removes a predicted decision when its own evidence contains suggestion/hedging language without a commitment cue. The goal is to mitigate the declared failure without replacing Gemini's reasoning.

In [ ]:
if ABLATION_RESULTS:
    rows=[]
    for case in EVAL_CASES:
        raw=ABLATION_RESULTS["C"][case["id"]]
        brief,removed=mitigate_decisions(raw["brief"]) if raw.get("valid") else ({},[])
        r=dict(raw)
        r["brief"]=brief
        r["valid"],r["validation_errors"]=validate_brief(brief) if brief else (False,["invalid"])
        s=score_case(case,r)
        s["removed_decisions"]=len(removed)
        rows.append(s)
    MITIGATION_DF=pd.DataFrame(rows)
    display(MITIGATION_DF.round(3))
    print("Configuration C mean decision F1:", round(METRICS[METRICS.config=="C"].decision_f1.mean(),3))
    print("Mitigated mean decision F1:", round(MITIGATION_DF.decision_f1.mean(),3))

## Optional small local-model comparison

The final architecture does not depend on this section. If a CPU-capable llama.cpp server is available, Qwen3.5-0.8B can be measured on the narrow five-label span-classification task. This is optional because the capstone rubric requires an integrated working system, not a local-model benchmark.

In [ ]:
if os.getenv("RUN_LOCAL_MODEL","0") != "1":
    print("Local comparison skipped. Set RUN_LOCAL_MODEL=1 only if a llama.cpp endpoint is available.")
else:
    LOCAL_DF=run_local_comparison()
    display(LOCAL_DF)
    print("Local classification accuracy:", round(LOCAL_DF.correct.mean(),3))
    print("Mean local latency:", round(LOCAL_DF.latency_s.mean(),3), "seconds")

## Submission guard

Do not submit until the notebook has been executed and the final cell reports `READY FOR SUBMISSION`.

After the measured run:

1. Save the executed notebook with outputs.
2. Commit the populated `data/response_cache.json`.
3. Update `research-note.md` with the actual measurements and model-selection decision.
4. Export the executed notebook to HTML and commit it.
5. Submit the capstone PR URL and repository URL in Canvas.

In [ ]:
problems=[]
if not ABLATION_RESULTS:
    problems.append("Ablation results missing")
if "JUDGE_DF" not in globals() or len(JUDGE_DF) < len(EVAL_CASES) * 3:
    problems.append("LLM-as-judge results missing/incomplete")
if FAILURE is None:
    problems.append("Failure trace missing")
if not pathlib.Path("data/response_cache.json").exists():
    problems.append("Persistent response cache missing")

if problems:
    print("NOT READY FOR SUBMISSION")
    for p in problems:
        print("-", p)
else:
    print("READY FOR SUBMISSION")
    print("- 12 held-out cases evaluated under A/B/C")
    print("- deterministic metrics reported")
    print("- Gemini judge completed on 3 named dimensions")
    print("- failure traced to a component")
    print("- mitigation measured")
    print("- API responses cached for reruns")

print("\nExport after execution:")
print("jupyter nbconvert --to html week8_meeting_to_action_capstone.ipynb")